# ==============================================================================
# A-EYEHORUS SENTINEL - SPRINT 4
# Módulo de Integração com LLM (Groq API / Llama-3)
# Objetivo: Gerar recomendações prescritivas reais para incidentes críticos.
# ==============================================================================

In [57]:
# PASSO 1: Instalar a biblioteca da Groq
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import oracledb
import pandas as pd
from dotenv import load_dotenv
from groq import Groq
from sqlalchemy import create_engine
from sqlalchemy.dialects.oracle import NUMBER, TIMESTAMP, VARCHAR2

load_dotenv()
warnings.filterwarnings('ignore')

In [58]:
# PASSO 2: Configurar a Chave da API
# Inicializa o cliente da Groq
API_KEY = os.getenv("API_GROK_KEY")
client = Groq(api_key=API_KEY)

In [3]:
# PASSO 3: Carregar a base de alertas gerada pelo seu XGBoost
# Substitua pelo nome do DataFrame que você já tem no seu Jupyter
# Exemplo: df_alertas = df_alertas_criticos.copy()
df_alertas = pd.read_csv('TB_ALERTAS_ITSM.csv')

# Para o MVP e Vídeo Pitch, vamos filtrar apenas os 10 incidentes mais críticos
# (Maior tempo de duração ou prioridade alta) para não estourar o limite gratuito da API
df_top_alertas = df_alertas.sample(10).copy()

In [4]:
# PASSO 4: Função para conversar com o Llama-3
def gerar_recomendacao_aiops(equipe, categoria, prioridade):
    """
    Envia o contexto do incidente para o Llama-3 e retorna a recomendação prescritiva.
    """
    
    # O Prompt de Sistema (Define quem a IA é e como deve se comportar)
    system_prompt = """
    Você é o 'A-EYEHORUS', uma Inteligência Artificial AIOps de nível Sênior atuando na Locaweb.
    Analise o alerta e forneça uma recomendação clara, direta e prescritiva.
    Aja como um gerente técnico orientando sua equipe. Indique qual deve ser a prioridade.
    Não use saudações, apenas forneça a ação. Mantenha o texto em no máximo 3 frases curtas.
    """
    
    # O Prompt do Usuário (Os dados reais do incidente detectado)
    user_prompt = f"""
    Alerta Crítico: Risco de quebra de OLA iminente.
    - Equipe Responsável: {equipe}
    - Categoria do Incidente: {categoria}
    - Prioridade: {prioridade}
    
    Prescreva a ação imediata.
    """
    
    try:
        # Fazendo a chamada para a API da Groq (usando o Llama 3 8B, super rápido e leve)
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            model="openai/gpt-oss-120b",
            temperature=0.6, # Temperatura baixa para respostas mais focadas e menos criativas
            max_tokens=150
        )
        
       # Extrai a resposta
        resposta = chat_completion.choices[0].message.content
        
        # Tratativa de segurança caso a resposta venha vazia
        if resposta is None or resposta.strip() == "":
            return f"ALERTA A-EYEHORUS: Risco de quebra de OLA na equipe {equipe}. Escalar e priorizar fila no Kanban."
        else:
            return f"ALERTA A-EYEHORUS: {resposta.strip()}"
        
    except Exception as e:
        print(f"Erro na API Groq: {e}")
        return f"ALERTA A-EYEHORUS: Risco sistêmico ({categoria}). Investigar equipe {equipe} urgentemente."

In [5]:
# PASSO 5: Aplicar a Inteligência na sua base de dados
print("Iniciando o War Room A-EYEHORUS (Conectando ao Llama-3)...")

recomendacoes = []
for index, row in df_top_alertas.iterrows():
    # Extrai os dados da linha atual
    equipe = row.get('NM_GRUPO_DESIGNADO', 'N/A')
    categoria = row.get('NM_CATEGORIA', 'N/A')
    prioridade = row.get('TP_PRIORIDADE', 'N/A')
    
    print(f"Analisando incidente da equipe {equipe} (Categoria: {categoria})...")
    
    # Chama a IA
    texto_llm = gerar_recomendacao_aiops(equipe, categoria, prioridade)
    recomendacoes.append(texto_llm)
    print(f"Recomendação gerada: {texto_llm}\n")
    # Pequena pausa para respeitar limites da API gratuita
    time.sleep(1) 

# Adiciona a nova coluna com as respostas reais da IA
df_top_alertas['TEXTO_PRESCRITIVO_LLM'] = recomendacoes

# PASSO 6: Salvar o resultado para o Power BI
df_top_alertas.to_csv('TB_ALERTAS_ITSM_COM_IA.csv', index=False)

print("\nConcluído! A base TB_ALERTAS_ITSM_COM_IA.csv foi gerada.")
print("Exemplo de resposta gerada pela IA:")
print(df_top_alertas['TEXTO_PRESCRITIVO_LLM'].iloc[0])

Iniciando o War Room A-EYEHORUS (Conectando ao Llama-3)...
Analisando incidente da equipe Team14 (Categoria: cat64)...
Recomendação gerada: ALERTA A-EYEHORUS: Risco de quebra de OLA na equipe Team14. Escalar e priorizar fila no Kanban.

Analisando incidente da equipe Team05 (Categoria: cat76)...
Recomendação gerada: ALERTA A-EYEHORUS: Risco de quebra de OLA na equipe Team05. Escalar e priorizar fila no Kanban.

Analisando incidente da equipe Team11 (Categoria: cat85)...
Recomendação gerada: ALERTA A-EYEHORUS: Risco de quebra de OLA na equipe Team11. Escalar e priorizar fila no Kanban.

Analisando incidente da equipe Team01 (Categoria: Nao Informado)...
Recomendação gerada: ALERTA A-EYEHORUS: Risco de quebra de OLA na equipe Team01. Escalar e priorizar fila no Kanban.

Analisando incidente da equipe Team14 (Categoria: cat41)...
Recomendação gerada: ALERTA A-EYEHORUS: Risco de quebra de OLA na equipe Team14. Escalar e priorizar fila no Kanban.

Analisando incidente da equipe Team11 (Cate

In [6]:
resultado = pd.read_csv('TB_ALERTAS_ITSM_COM_IA.csv')
resultado.head(10)

,CD_INCIDENTE,DT_ABERTO,TP_PRIORIDADE,NM_CATEGORIA,NM_GRUPO_DESIGNADO,RISCO_SLA_PREVISTO,STATUS_REAL,TEXTO_PRESCRITIVO_LLM
0,INC8536630,2025-09-16 13:30:55,3 - Média,cat64,Team14,1,1,ALERTA A-EYEHORUS: Risco de quebra de OLA na e...
1,INC8535973,2025-09-15 19:02:44,3 - Média,cat76,Team05,1,0,ALERTA A-EYEHORUS: Risco de quebra de OLA na e...
2,INC8547537,2025-09-26 09:46:31,3 - Média,cat85,Team11,1,0,ALERTA A-EYEHORUS: Risco de quebra de OLA na e...
3,INC8552388,2025-10-01 14:11:10,3 - Média,Nao Informado,Team01,1,0,ALERTA A-EYEHORUS: Risco de quebra de OLA na e...
4,INC8536575,2025-09-16 12:27:05,3 - Média,cat41,Team14,1,1,ALERTA A-EYEHORUS: Risco de quebra de OLA na e...
5,INC8543741,2025-09-22 18:14:29,3 - Média,cat24,Team11,1,1,ALERTA A-EYEHORUS: Risco de quebra de OLA na e...
6,INC8557556,2025-10-06 10:19:35,3 - Média,cat73,Team09,1,1,ALERTA A-EYEHORUS: Risco de quebra de OLA na e...
7,INC8557618,2025-10-06 11:17:50,3 - Média,cat97,Team05,1,0,ALERTA A-EYEHORUS: Risco de quebra de OLA na e...
8,INC8539231,2025-09-19 09:51:43,2 - Alta,cat76,Team05,1,0,ALERTA A-EYEHORUS: Risco de quebra de OLA na e...
9,INC8535624,2025-09-15 11:57:16,3 - Média,cat31,Team11,1,0,ALERTA A-EYEHORUS: Risco de quebra de OLA na e...


In [7]:
resultado['TEXTO_PRESCRITIVO_LLM'].head(10)

0    ALERTA A-EYEHORUS: Risco de quebra de OLA na e...
1    ALERTA A-EYEHORUS: Risco de quebra de OLA na e...
2    ALERTA A-EYEHORUS: Risco de quebra de OLA na e...
3    ALERTA A-EYEHORUS: Risco de quebra de OLA na e...
4    ALERTA A-EYEHORUS: Risco de quebra de OLA na e...
5    ALERTA A-EYEHORUS: Risco de quebra de OLA na e...
6    ALERTA A-EYEHORUS: Risco de quebra de OLA na e...
7    ALERTA A-EYEHORUS: Risco de quebra de OLA na e...
8    ALERTA A-EYEHORUS: Risco de quebra de OLA na e...
9    ALERTA A-EYEHORUS: Risco de quebra de OLA na e...
Name: TEXTO_PRESCRITIVO_LLM, dtype: str

# Jogar para o ADB

In [50]:
resultado['DT_ABERTO'] = pd.to_datetime(resultado['DT_ABERTO'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

In [51]:
resultado.dtypes

CD_INCIDENTE                        str
DT_ABERTO                datetime64[us]
TP_PRIORIDADE                       str
NM_CATEGORIA                        str
NM_GRUPO_DESIGNADO                  str
RISCO_SLA_PREVISTO                int64
STATUS_REAL                       int64
TEXTO_PRESCRITIVO_LLM               str
dtype: object

In [52]:
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
CONNECT_STRING = '(description= (retry_count=20)(retry_delay=3)(address=(protocol=tcps)(port=1522)(host=adb.sa-saopaulo-1.oraclecloud.com))(connect_data=(service_name=g253a13c6f2211a_dadosaiopslocaweb_low.adb.oraclecloud.com))(security=(ssl_server_dn_match=yes)))'

def conectar_oracle():
    return oracledb.connect(
        user=DB_USER,
        password=DB_PASSWORD,
        dsn=CONNECT_STRING,
        wallet_location=r"C:/opt/OracleCloud/Wallet_dadosAIOpsLocaweb",
        wallet_password=DB_PASSWORD
    )

In [53]:
# Criando a conexão
engine = create_engine("oracle+oracledb://", creator = conectar_oracle)
print("Iniciando upload para o Oracle Cloud...")

tipagem_oracle = {
    'CD_INCIDENTE': VARCHAR2(50),
    'DT_ABERTO': TIMESTAMP(timezone=False),
    'TP_PRIORIDADE': VARCHAR2(20),
    'NM_CATEGORIA': VARCHAR2(100),
    'NM_GRUPO_DESIGNADO': VARCHAR2(100),
    'RISCO_SLA_PREVISTO': NUMBER(precision= 15, scale = 0),
    'STATUS_REAL': NUMBER(precision= 15, scale = 0),
    'TEXTO_PRESCRITIVO_LLM': VARCHAR2(4000)
}
# Enviando a tabela para o Oracle Cloud
try:
    resultado.to_sql(
        'TB_ALERTAS_ITSM_IA',
        con=engine,
        if_exists='replace',
        dtype=tipagem_alertas[0],  # remove o efeito da vírgula na célula anterior
        index=False
    )
    print("- TB_ALERTAS_ITSM inserida com sucesso!")
except Exception as e:
    print(f"Erro ao enviar dados para o Oracle Cloud: {e}")

Iniciando upload para o Oracle Cloud...
- TB_ALERTAS_ITSM inserida com sucesso!
